In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score

import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [6]:
load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

In [7]:
with engine.connect() as connection:
    print("Connected!")

Connected!


In [10]:
dataset_info_query = """
select
    min(transaction_timestamp) as min_date,
    max(transaction_timestamp) as max_date,
    count(*) as number_of_transactions
from analytics.fraud_ml_features
"""

dataset_info = pd.read_sql(dataset_info_query, engine)

dataset_info

,min_date,max_date,number_of_transactions
0,2010-01-01 00:01:00,2019-10-31 23:57:00,8914963


In [12]:
fraud_by_year_query = """
select
    extract(year from transaction_timestamp)::int as year,
    count(*) as transactions,
    count(*) filter (where is_fraud = true) as fraud_transactions,
    round(
        count(*) filter (where is_fraud = true)::numeric
        / count(*) * 100,
        4
    ) as fraud_rate
from analytics.fraud_ml_features
group by extract(year from transaction_timestamp)
order by year
"""

fraud_by_year = pd.read_sql(fraud_by_year_query, engine)

fraud_by_year

,year,transactions,fraud_transactions,fraud_rate
0,2010,831529,2573,0.3094
1,2011,863428,37,0.0043
2,2012,885421,923,0.1042
3,2013,907304,1337,0.1474
4,2014,915073,664,0.0726
5,2015,930224,2189,0.2353
6,2016,932762,2448,0.2624
7,2017,937284,172,0.0184
8,2018,934599,1629,0.1743
9,2019,777339,1360,0.1750


In [14]:
fraud_by_month_query = """
select
    date_trunc('month', transaction_timestamp) as month,
    count(*) as transactions,
    count(*) filter (where is_fraud = true) as fraud_transactions,
    round(
        count(*) filter (where is_fraud = true)::numeric
        / count(*) * 100,
        4
    ) as fraud_rate
from analytics.fraud_ml_features
group by date_trunc('month', transaction_timestamp)
order by month
"""

fraud_by_month = pd.read_sql(fraud_by_month_query, engine)

fraud_by_month

count    118.000000
mean     112.983051
std       99.149717
min        0.000000
25%        0.000000
50%      116.000000
75%      192.750000
max      423.000000
Name: fraud_transactions, dtype: float64

In [15]:
fraud_by_month["fraud_transactions"].describe()

count    118.000000
mean     112.983051
std       99.149717
min        0.000000
25%        0.000000
50%      116.000000
75%      192.750000
max      423.000000
Name: fraud_transactions, dtype: float64

In [16]:
print("Months with 0 fraud:",
      (fraud_by_month["fraud_transactions"] == 0).sum())

print("Months with < 10 fraud:",
      (fraud_by_month["fraud_transactions"] < 10).sum())

print("Min fraud rate:",
      fraud_by_month["fraud_rate"].min())

print("Max fraud rate:",
      fraud_by_month["fraud_rate"].max())

Months with 0 fraud: 39
Months with < 10 fraud: 40
Min fraud rate: 0.0
Max fraud rate: 0.5278


In [17]:
zero_fraud_months = fraud_by_month[
    fraud_by_month["fraud_transactions"] == 0
]

zero_fraud_months

,month,transactions,fraud_transactions,fraud_rate
13,2011-02-01,65223,0,0.0
14,2011-03-01,72224,0,0.0
15,2011-04-01,70777,0,0.0
16,2011-05-01,72667,0,0.0
17,2011-06-01,71162,0,0.0
18,2011-07-01,73440,0,0.0
19,2011-08-01,74420,0,0.0
20,2011-09-01,71418,0,0.0
21,2011-10-01,73877,0,0.0
22,2011-11-01,71564,0,0.0


In [18]:
train_query = """
(
    select *
    from analytics.fraud_ml_features
    where transaction_timestamp < '2018-01-01'
      and is_fraud = true
)
union all
(
    select *
    from analytics.fraud_ml_features
    where transaction_timestamp < '2018-01-01'
      and is_fraud = false
    order by random()
    limit 300000
)
"""

train_df = pd.read_sql(train_query, engine)

train_df.shape

(310343, 16)

In [19]:
train_df["is_fraud"].value_counts()

is_fraud
False    300000
True      10343
Name: count, dtype: int64

In [20]:
train_df["is_fraud"].value_counts(normalize=True)

is_fraud
False    0.966672
True     0.033328
Name: proportion, dtype: float64

In [21]:
train_df.columns.tolist()

['transaction_key',
 'transaction_timestamp',
 'amount',
 'mcc_key',
 'use_chip',
 'merchant_id',
 'transaction_hour',
 'day_of_week',
 'is_weekend',
 'user_previous_transaction_count',
 'user_previous_avg_amount',
 'amount_vs_user_avg',
 'card_previous_transaction_count',
 'card_previous_avg_amount',
 'amount_vs_card_avg',
 'is_fraud']

In [22]:
X_train = train_df.drop(
    columns=[
        "transaction_key",
        "transaction_timestamp",
        "is_fraud"
    ]
)

y_train = train_df["is_fraud"]

In [23]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (310343, 13)
y_train: (310343,)


In [24]:
val_query = """
select *
from analytics.fraud_ml_features
where transaction_timestamp >= '2018-01-01'
  and transaction_timestamp < '2019-01-01'
"""

val_df = pd.read_sql(val_query, engine)

val_df.shape

(934599, 16)

In [25]:
X_val = val_df.drop(
    columns=[
        "transaction_key",
        "transaction_timestamp",
        "is_fraud"
    ]
)

y_val = val_df["is_fraud"]

In [26]:
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nValidation target:")
print(y_val.value_counts())

print("\nValidation proportions:")
print(y_val.value_counts(normalize=True))

X_val: (934599, 13)
y_val: (934599,)

Validation target:
is_fraud
False    932970
True       1629
Name: count, dtype: int64

Validation proportions:
is_fraud
False    0.998257
True     0.001743
Name: proportion, dtype: float64


In [27]:
numeric_features = [
    "amount",
    "user_previous_transaction_count",
    "user_previous_avg_amount",
    "amount_vs_user_avg",
    "card_previous_transaction_count",
    "card_previous_avg_amount",
    "amount_vs_card_avg"
]

categorical_features = [
    "mcc_key",
    "use_chip",
    "merchant_id",
    "transaction_hour",
    "day_of_week"
]

binary_features = [
    "is_weekend"
]

In [28]:
selected_features = numeric_features + categorical_features + binary_features

print("Features in X_train:", X_train.shape[1])
print("Features selected:", len(selected_features))

set(X_train.columns) - set(selected_features)

Features in X_train: 13
Features selected: 13


set()

In [29]:
X_train.isnull().sum()

amount                              0
mcc_key                             0
use_chip                            0
merchant_id                         0
transaction_hour                    0
day_of_week                         0
is_weekend                          0
user_previous_transaction_count     0
user_previous_avg_amount           32
amount_vs_user_avg                 32
card_previous_transaction_count     0
card_previous_avg_amount           93
amount_vs_card_avg                 93
dtype: int64

In [30]:
X_train.dtypes

amount                             float64
mcc_key                              int64
use_chip                               str
merchant_id                          int64
transaction_hour                   float64
day_of_week                        float64
is_weekend                           int64
user_previous_transaction_count      int64
user_previous_avg_amount           float64
amount_vs_user_avg                 float64
card_previous_transaction_count      int64
card_previous_avg_amount           float64
amount_vs_card_avg                 float64
dtype: object

In [39]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [35]:
categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="passthrough"
)

In [41]:
dummy_model = DummyClassifier(strategy="most_frequent")

In [43]:
dummy_model.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.97,0.03]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[bool](2,)","[False, True]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](13,)","['amount','mcc_key','use_chip',...,'card_previous_transaction_count', 'card_previous_avg_amount','amount_vs_card_avg']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,13
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [44]:
y_val_pred_dummy = dummy_model.predict(X_val)

In [45]:
np.unique(y_val_pred_dummy, return_counts=True)

(array([False]), array([934599]))

In [48]:
dummy_accuracy = accuracy_score(y_val, y_val_pred_dummy)

dummy_accuracy

0.9982570064808544

In [50]:
dummy_confusion_matrix = confusion_matrix(
    y_val,
    y_val_pred_dummy
)

dummy_confusion_matrix

array([[932970,      0],
       [  1629,      0]])

In [54]:
dummy_recall = recall_score(y_val, y_val_pred_dummy)

dummy_recall

0.0

In [56]:
dummy_precision = precision_score(
    y_val,
    y_val_pred_dummy,
    zero_division=0
)

dummy_precision

0.0

In [58]:
dummy_f1 = f1_score(
    y_val,
    y_val_pred_dummy,
    zero_division=0
)

dummy_f1

0.0

In [59]:
dummy_model.predict(X_val)

array([False, False, False, ..., False, False, False], shape=(934599,))

In [60]:
dummy_model.predict_proba(X_val[:5])

array([[1., 0.],
       [1., 0.],
       [1., 0.],
       [1., 0.],
       [1., 0.]])

In [61]:
dummy_model.classes_

array([False,  True])

In [65]:
y_val_proba = dummy_model.predict_proba(X_val)[:, 1]

y_val_proba

array([0., 0., 0., ..., 0., 0., 0.], shape=(934599,))